In [1]:
# Import PyTorch neural networks
import torch
import torch.nn as nn
import numpy as np

# Step 1: Create the Actor network
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action=1.0):
        super(Actor, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()   # Restricts output between -1 and 1
        )

        self.max_action = max_action

    def forward(self, state):
        return self.max_action * self.fc(state)
        
# Step 2: Create the Critic network
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()

        self.l1 = nn.Linear(state_dim + action_dim, 64)
        self.l2 = nn.Linear(64, 1)

    def forward(self, state, action):
        inputs = torch.cat([state, action], dim=-1)
        x = torch.relu(self.l1(inputs))
        return self.l2(x)

# Step 3: Soft Update
def soft_update(target_net, local_net, tau=0.005):
    for target_param, local_param in zip(target_net.parameters(),
                                         local_net.parameters()):
        target_param.data.copy_(
            tau * local_param.data +
            (1.0 - tau) * target_param.data
        )

# Step 4: OU Noise
class OUNoise:
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.size = size
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.state = np.ones(size) * mu

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return self.state

# ------------------------------------------------------------
# Initialize Networks
# ------------------------------------------------------------

actor = Actor(state_dim=3, action_dim=1, max_action=2.0)
target_actor = Actor(state_dim=3, action_dim=1, max_action=2.0)
critic = Critic(state_dim=3, action_dim=1)

# Copy weights initially
soft_update(target_actor, actor, tau=1.0)

# Initialize OU Noise
noise = OUNoise(size=1)

# Dummy state
state = torch.randn((1, 3))

print("DDPG Learning Demonstration\n")

# ------------------------------------------------------------
# Simulate 3 Learning Steps
# ------------------------------------------------------------

for step in range(1, 4):

    # Actor predicts action
    action = actor(state)

    # Add exploration noise
    noisy_action = action + torch.tensor(
        noise.sample(), dtype=torch.float32)

# Critic evaluates action
    q_value = critic(state, noisy_action).item()

    print(f"Step {step}")
    print(f"Actor Action     : {action.item():.4f}")
    print(f"Action + OU Noise: {noisy_action.item():.4f}")
    print(f"Critic Q-Value   : {q_value:.4f}")

    # Simulate learning by slightly changing Actor weights
    with torch.no_grad():
        for param in actor.parameters():
            param += 0.02 * torch.randn_like(param)

    # Slowly update target network
    soft_update(target_actor, actor, tau=0.1)

    print("-" * 40)

print("DOPG Continuous Actor-Critic setup successfully initialized")    

DDPG Learning Demonstration

Step 1
Actor Action     : 0.4961
Action + OU Noise: 0.8709
Critic Q-Value   : 0.3214
----------------------------------------
Step 2
Actor Action     : 0.8662
Action + OU Noise: 1.0981
Critic Q-Value   : 0.3089
----------------------------------------
Step 3
Actor Action     : 0.7218
Action + OU Noise: 0.6835
Critic Q-Value   : 0.3398
----------------------------------------
DOPG Continuous Actor-Critic setup successfully initialized
